# AmsiNet - Step 7: word agent v2 (rectangular input, distillation from the flat ViT-B teacher, logit-adjusted loss, EMA)

**Runtime:** Colab **A100**. Needs Steps 1-3a: `bstd_index.json`, `crops_meta.csv`, `crops224.tar`, the flat teacher checkpoint `ckpt/vit_base_patch16_224_seed0.pt`, and `agents/word_moe_seed0.json` (for the expert groups). The first run re-cuts the word crops at a rectangular size from the original images (needs `data/detection.zip` on Drive, about 10-15 minutes, resumable).

Why v2 (what the v1 word agent was missing)
1. **Square 224x224 letterbox wastes pixels**: a typical word is 3-5 times wider than high, so most of the square is padding and the glyphs are tiny. Rectangular 64x192 gives the text roughly 2x the useful pixels at lower cost.
2. **Distillation from the flat ViT-B/16 teacher** (the strongest single model we have): soft targets with temperature, so the 3-4M-parameter student inherits the teacher's decision structure.
3. **Logit-adjusted loss** (training-prior correction inside the loss): removes the Latin sink that hurts macro-F1.
4. **Better recipe**: longer training, higher learning rate for a small model, stronger augmentation, EMA weights.

Everything stays within protocol: validation picks the epoch, test is touched once. The saved probabilities use the same crop order as `crops_meta.csv`, so Step 6 can fuse them directly (`WORD_TAG = 'wordv2_<backbone>_seed0'`).

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
!pip -q install timm gdown
import torch, timm
print('torch', torch.__version__, '| timm', timm.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to GPU)')

In [ ]:
# 2. Configuration, index, restore of the square crops
import json, os, glob, math, random, time, copy, subprocess
import numpy as np, pandas as pd
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
Image.MAX_IMAGE_PIXELS = None
ROOT, LOCAL = '/content/drive/MyDrive/amsinet', '/content/bstd'
os.makedirs(f'{ROOT}/agents', exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

CFG = dict(SEEDS=[0], BACKBONE='efficientvit_b0.r224_in1k',   # other light options: 'efficientvit_b1.r224_in1k', 'efficientvit_b2.r224_in1k'
           HW=(64, 192),                                       # rectangular word input (height, width); try (80, 240) if time allows
           EPOCHS=30, BATCH=256, LR=5e-4, WD=0.05, WORKERS=8, EMA=0.999,
           MOE=True, LAMBDA_GATE=0.3, SMOOTH=0.05,
           KD_ALPHA=0.5, KD_T=2.0,                             # KD_ALPHA = 0 switches distillation off (ablation)
           LA_TAU=0.5,                                         # logit adjustment strength in the loss (0 = off)
           PAD=0.04, TEACHER='vit_base_patch16_224_seed0', PRINT_TEST=False)   # PRINT_TEST stays False while we choose settings on validation
H, W = CFG['HW']; FILL = (124, 116, 104); DEV = 'cuda'
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
IMG_ROOT, SCRIPTS, INDEX = D['img_root'], D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS)
IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv')
CROPS, CROPS_TAR = f'{LOCAL}/crops224', f'{ROOT}/data/crops224.tar'
if not os.path.isdir(CROPS) and os.path.exists(CROPS_TAR):
    print('restoring', CROPS_TAR); subprocess.run(['tar', '-xf', CROPS_TAR, '-C', LOCAL], check=True)
GROUP = json.load(open(f'{ROOT}/agents/word_moe_seed0.json'))['groups']; M = max(GROUP.values()) + 1
print(len(META), 'word crops |', M, 'expert groups |', {g: [s for s in SCRIPTS if GROUP[s] == g] for g in range(M)})

## 3. Rectangular word crops (one-time, resumable)

In [ ]:
RECT, RECT_TAR = f'{LOCAL}/rect{H}x{W}', f'{ROOT}/data/rect{H}x{W}.tar'
if not os.path.isdir(RECT) and os.path.exists(RECT_TAR):
    print('restoring', RECT_TAR); subprocess.run(['tar', '-xf', RECT_TAR, '-C', LOCAL], check=True)

def letterbox_rect(im, w, h):
    iw, ih = im.size; s = min(w / iw, h / ih); nw, nh = max(1, int(round(iw * s))), max(1, int(round(ih * s)))
    im = im.resize((nw, nh), Image.BICUBIC, reducing_gap=3.0); cv = Image.new('RGB', (w, h), FILL); cv.paste(im, ((w - nw) // 2, (h - nh) // 2)); return cv

def ensure_images():
    global IMG_ROOT
    if os.path.isdir(IMG_ROOT): return
    z, zdrive = f'{LOCAL}/detection.zip', f'{ROOT}/data/detection.zip'
    if not os.path.exists(z):
        assert os.path.exists(zdrive), 'data/detection.zip is not on Drive: run Step 3a once (it downloads and saves it), then come back'
        print('copying detection.zip from Drive (~5-10 min)...'); subprocess.run(['cp', zdrive, z], check=True)
    subprocess.run(['unzip', '-q', '-o', z, '-d', LOCAL], check=True)
    IMG_ROOT = os.path.dirname(glob.glob(f'{LOCAL}/**/BSTD_v17.57.json', recursive=True)[0])

META['rpath'] = [f'{RECT}/{sp}/{iid}_{k}.jpg' for sp, iid, k in zip(META.split, META.image_id, META.word_idx)]
todo = META[~META.rpath.map(os.path.exists)]
print('rectangular crops missing:', len(todo), 'of', len(META))
if len(todo):
    ensure_images(); groups = list(todo.groupby('image_id')); BAD = []
    def work(item):
        iid, g = item; rec = IDX[iid]; im = ImageOps.exif_transpose(Image.open(os.path.join(IMG_ROOT, rec['image']))).convert('RGB')
        for r in g.itertuples():
            x1, y1, x2, y2 = rec['words'][r.word_idx]['bbox']; px, py = (x2 - x1) * CFG['PAD'], (y2 - y1) * CFG['PAD']
            box = (max(0, int(x1 - px)), max(0, int(y1 - py)), min(im.width, int(math.ceil(x2 + px))), min(im.height, int(math.ceil(y2 + py))))
            if box[2] - box[0] < 2 or box[3] - box[1] < 2: BAD.append(r.rpath); continue
            os.makedirs(os.path.dirname(r.rpath), exist_ok=True); letterbox_rect(im.crop(box), W, H).save(r.rpath, quality=95)
    with ThreadPoolExecutor(6) as ex:
        for i, _ in enumerate(ex.map(work, groups)):
            if (i + 1) % 500 == 0: print(f'{i + 1}/{len(groups)} images')
    print('skipped:', len(BAD)); assert not BAD, 'some rectangular crops could not be cut; crops_meta.csv and the crop rule disagree'
    subprocess.run(['tar', '-cf', RECT_TAR, '-C', LOCAL, f'rect{H}x{W}'], check=True)
print('rectangular crops ready:', META.rpath.map(os.path.exists).all())

## 4. Teacher logits (flat ViT-B/16 from Step 2) on the training crops
The teacher sees the same 224x224 square crops it was trained on. Its training-set predictions are confident (it has seen these words), so the distillation signal is mostly its *relative* class scores, not new information about hard words; the temperature softens this. The ablation `KD_ALPHA = 0` quantifies how much the teacher actually helps.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import f1_score
TL_PATH = f"{ROOT}/agents/teacher_{CFG['TEACHER']}_train_logits.npy"
if not os.path.exists(TL_PATH):
    tm = timm.create_model('vit_base_patch16_224.augreg_in21k', pretrained=False, num_classes=C)
    tm.load_state_dict(torch.load(f"{ROOT}/ckpt/{CFG['TEACHER']}.pt", map_location='cpu')); tm = tm.to(DEV).eval()
    mc = timm.data.resolve_data_config({}, model=tm); tf = T.Compose([T.ToTensor(), T.Normalize(mc['mean'], mc['std'])])
    class SQ(Dataset):
        def __init__(s, p): s.p = p
        def __len__(s): return len(s.p)
        def __getitem__(s, i): return tf(Image.open(s.p[i]).convert('RGB'))
    tr = META[META.split == 'train']; out = []
    with torch.no_grad():
        for x in DataLoader(SQ(tr.path.tolist()), batch_size=256, num_workers=CFG['WORKERS']):
            with torch.autocast('cuda', dtype=torch.float16): out.append(tm(x.to(DEV)).float().cpu())
    np.save(TL_PATH, torch.cat(out).numpy().astype(np.float16)); del tm
TL = np.load(TL_PATH).astype(np.float32); TR = META[META.split == 'train']
assert len(TL) == len(TR)
print('teacher logits:', TL.shape, '| teacher accuracy on its own training words: %.3f' % (TL.argmax(1) == np.array([SIDX[s] for s in TR.script])).mean())

## 5. Model, loss and training loop

In [ ]:
GRP_T = torch.tensor([GROUP[s] for s in SCRIPTS])
PRIOR = np.bincount([SIDX[s] for s in TR.script], minlength=C).astype(np.float32); PRIOR /= PRIOR.sum()
LOGPRIOR = torch.tensor(np.log(PRIOR))

class MoEHead(nn.Module):
    def __init__(s, d, ncls, M, hid=256):
        super().__init__(); s.gate = nn.Linear(d, M)
        s.experts = nn.ModuleList([nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, ncls)) for _ in range(M)])
    def forward(s, f):
        g = s.gate(f).softmax(-1); z = torch.stack([e(f) for e in s.experts], 1)
        return (g.unsqueeze(-1) * z).sum(1), g

class WordAgent(nn.Module):
    def __init__(s, name, moe):
        super().__init__(); s.bb = timm.create_model(name, pretrained=True, num_classes=0); s.bb.eval()
        with torch.no_grad(): d = s.bb(torch.zeros(2, 3, H, W)).shape[1]      # measured at the rectangular size
        s.bb.train(); s.head = MoEHead(d, C, M) if moe else nn.Linear(d, C)
    def forward(s, x):
        f = s.bb(x); o = s.head(f); return o if isinstance(o, tuple) else (o, None)

class DS(Dataset):
    def __init__(s, paths, y, tl, tf): s.p, s.y, s.tl, s.tf = paths, y, tl, tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i], (s.tl[i] if s.tl is not None else np.zeros(C, np.float32))

def metrics(P, y):
    pr = P.argmax(1); return dict(acc=float((pr == y).mean()), macro_f1=float(f1_score(y, pr, average='macro', labels=list(range(C)), zero_division=0)),
                                   per_class_f1=f1_score(y, pr, average=None, labels=list(range(C)), zero_division=0).tolist())

@torch.no_grad()
def predict(model, dl):
    model.eval(); P = []
    for x, _, _ in dl:
        with torch.autocast('cuda', dtype=torch.float16): lo, _ = model(x.to(DEV, non_blocking=True))
        P.append(lo.float().softmax(1).cpu())
    return torch.cat(P).numpy()

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def fit_v2(seed, **over):
    c = {**CFG, **over}
    short = c['BACKBONE'].split('.')[0] + ('_moe' if c['MOE'] else '')
    tag = f"wordv2_{short}_kd{c['KD_ALPHA']}_la{c['LA_TAU']}_lr{c['LR']}_ep{c['EPOCHS']}_seed{seed}".replace('0.', '0p').replace('e-0', 'em'); out = f'{ROOT}/agents/{tag}.json'
    if os.path.exists(out): print('skip (done):', tag); return tag
    set_seed(seed); model = WordAgent(c['BACKBONE'], c['MOE']).to(DEV); ema = copy.deepcopy(model).eval()
    for p in ema.parameters(): p.requires_grad_(False)
    mc = timm.data.resolve_data_config({}, model=model.bb); norm = T.Normalize(mc['mean'], mc['std'])
    ttf = T.Compose([T.RandomAffine(5, translate=(0.03, 0.05), scale=(0.9, 1.1), shear=5, fill=FILL), T.RandomPerspective(0.15, p=0.3, fill=FILL),
                     T.ColorJitter(0.4, 0.4, 0.3), T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.3), T.ToTensor(), norm])
    etf = T.Compose([T.ToTensor(), norm])
    ytr = np.array([SIDX[s] for s in META[META.split == 'train'].script]); yva = np.array([SIDX[s] for s in META[META.split == 'val'].script]); yte = np.array([SIDX[s] for s in META[META.split == 'test'].script])
    P = {sp: META[META.split == sp].reset_index(drop=True) for sp in ('train', 'val', 'test')}
    Ltr = DataLoader(DS(P['train'].rpath.tolist(), ytr, TL, ttf), batch_size=c['BATCH'], shuffle=True, num_workers=c['WORKERS'], pin_memory=True, drop_last=True)
    Lva = DataLoader(DS(P['val'].rpath.tolist(), yva, None, etf), batch_size=512, num_workers=c['WORKERS']); Lte = DataLoader(DS(P['test'].rpath.tolist(), yte, None, etf), batch_size=512, num_workers=c['WORKERS'])
    opt = torch.optim.AdamW(model.parameters(), lr=c['LR'], weight_decay=c['WD']); total = c['EPOCHS'] * len(Ltr); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda'); best, best_state, hist = -1, None, []; la = c['LA_TAU'] * LOGPRIOR.to(DEV); Tk = c['KD_T']; a = c['KD_ALPHA']
    for ep in range(c['EPOCHS']):
        model.train(); t0 = time.time(); rl = 0
        for x, y, tl in Ltr:
            x, y, tl = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True), tl.to(DEV, non_blocking=True)
            with torch.autocast('cuda', dtype=torch.float16):
                lo, g = model(x); z = lo.float() + la[None, :]                 # logit adjustment: the raw logits become prior-free
                t = F.one_hot(y, C).float() * (1 - c['SMOOTH']) + c['SMOOTH'] / C
                ce = -(t * z.log_softmax(1)).sum(1).mean(); loss = ce
                if a > 0:
                    kd = F.kl_div((z / Tk).log_softmax(1), (tl / Tk).softmax(1), reduction='batchmean') * Tk * Tk; loss = (1 - a) * ce + a * kd
                if g is not None: loss = loss + c['LAMBDA_GATE'] * F.nll_loss(torch.log(g.float() + 1e-8), GRP_T.to(DEV)[y])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step(); rl += loss.item()
            with torch.no_grad():
                d = min(c['EMA'], (1 + sched.last_epoch) / (10 + sched.last_epoch)); msd = model.state_dict()
                for k, ve in ema.state_dict().items():
                    if ve.dtype.is_floating_point: ve.mul_(d).add_(msd[k].detach(), alpha=1 - d)    # parameters AND batch-norm statistics
                    else: ve.copy_(msd[k])
        me, ml = metrics(predict(ema, Lva), yva), metrics(predict(model, Lva), yva); hist.append(dict(epoch=ep + 1, loss=rl / len(Ltr), ema_f1=me['macro_f1'], live_f1=ml['macro_f1'], ema_acc=me['acc'], live_acc=ml['acc']))
        print(f"{tag} ep{ep + 1}: loss {rl / len(Ltr):.4f} | val EMA acc {me['acc']:.4f} F1 {me['macro_f1']:.4f} | live acc {ml['acc']:.4f} F1 {ml['macro_f1']:.4f} ({time.time() - t0:.0f}s)")
        for nm, mv, mdl in (('ema', me, ema), ('live', ml, model)):
            if mv['macro_f1'] > best: best = mv['macro_f1']; best_state = {k: v.detach().cpu().clone() for k, v in mdl.state_dict().items()}; best_from = nm
    ema.load_state_dict(best_state); print(f'selected weights: {best_from} (validation macro-F1 {best:.4f})'); res = dict(tag=tag, seed=seed, cfg=c, history=hist, groups=GROUP, best_val_f1=best, selected=best_from)
    for sp, L, yy in (('val', Lva, yva), ('test', Lte, yte)):
        Pp = predict(ema, L); np.save(f'{ROOT}/agents/{tag}_{sp}_probs.npy', Pp.astype(np.float16)); P[sp][['path']].to_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv', index=False)
        res[sp] = metrics(Pp, yy)
    torch.save(best_state, f'{ROOT}/agents/{tag}.pt'); json.dump(res, open(out, 'w'))
    print(f"{tag}: best validation macro-F1 {best:.4f}" + (f" | TEST acc {res['test']['acc']:.4f} macro-F1 {res['test']['macro_f1']:.4f}" if c['PRINT_TEST'] else ' | test saved but not printed (sweep)')); return tag

# quick smoke test (about 1 minute): checks the backbone accepts the rectangular size before the long run
_m = WordAgent(CFG['BACKBONE'], CFG['MOE']).to(DEV)
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16): print('smoke test output:', _m(torch.zeros(2, 3, H, W, device=DEV))[0].shape, '(expected [2, 10])')
del _m

## 6. Train (about 15-25 minutes on an A100 for one seed). Run the cell, then go and do something else.

In [ ]:
# Settings are chosen on VALIDATION macro-F1 only. Test numbers are saved to disk but not printed until PRINT_TEST = True for the final run.
SWEEP = [dict(),                      # baseline: rect input, KD 0.5, logit adjustment 0.5, lr 5e-4, 30 epochs
         dict(KD_ALPHA=0.0),          # no distillation
         dict(LA_TAU=0.0)]            # no logit adjustment
TAGS = [fit_v2(0, **o) for o in SWEEP]          # about 3 x 17 minutes on an A100

## 7. Compare with v1, the flat teacher and the published numbers
If v2 does not improve over v1 on **validation**, do not tune on test; tell me and we change one lever at a time (`KD_ALPHA`, `LA_TAU`, `HW`, `BACKBONE`). Ablation runs: set `KD_ALPHA = 0` or `LA_TAU = 0` in the config cell and run section 6 again (each variant gets its own tag).

In [ ]:
rows = []
def add(name, j, key=None):
    r = json.load(open(j)); t = r['test'] if key is None else r['test'][key]; rows.append((name, round(100 * t['acc'], 2), round(100 * t['macro_f1'], 2)))
add('word agent v1 (EfficientViT-B0, square, MoE)', f'{ROOT}/agents/word_moe_seed0.json', 'word')
add('flat ViT-B/16 teacher', f"{ROOT}/results/{CFG['TEACHER'].replace('_seed0', '')}_seed0.json")
for tg in TAGS:
    if CFG['PRINT_TEST']: add('word agent v2: ' + tg, f'{ROOT}/agents/{tg}.json')      # test numbers stay hidden during the sweep
VAL = pd.DataFrame([(tg, round(100 * json.load(open(f'{ROOT}/agents/{tg}.json'))['best_val_f1'], 2), json.load(open(f'{ROOT}/agents/{tg}.json'))['selected']) for tg in TAGS], columns=['variant', 'best validation macro-F1 %', 'weights'])
display(VAL)   # choose the variant from THIS table, not from the test table below
CMP = pd.DataFrame(rows, columns=['model (BSTD test words, seed 0)', 'accuracy %', 'macro-F1 %']); display(CMP); CMP.to_csv(f'{ROOT}/results/word_v2_comparison.csv', index=False)

# cost of the v2 word agent at the rectangular input
from torch.utils.flop_counter import FlopCounterMode
m = WordAgent(CFG['BACKBONE'], CFG['MOE']).cpu().eval(); x = torch.randn(1, 3, H, W)
with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
with torch.no_grad():
    for _ in range(5): m(x)
    ts = []
    for _ in range(30): t0 = time.perf_counter(); m(x); ts.append((time.perf_counter() - t0) * 1000)
print('v2 word agent: params %.2f M | %.2f GFLOPs | CPU batch-1 latency %.1f ms (input %dx%d)' % (sum(p.numel() for p in m.parameters()) / 1e6, fc.get_total_flops() / 1e9, float(np.median(ts)), H, W))